# 🏭 EEIO 탄소배출 시뮬레이터

기업의 **매출액**과 **원가 구조**를 입력하면, 한국 2023년 KR_EEIO(환경산업연관표) 데이터를 기반으로  
**Scope 1 · 2 · 3 탄소배출량**을 자동으로 계산합니다.

여기서의 Scope3는 산업연관표를 이용한 **공급망 배출**을 의미합니다. 
따라서 구매한 원재료 뿐만 아니라 **운송, 출장, 폐기 등** 다양한 범주를 포함하는 **GHG Protocal**과 다릅니다.


> **필요 파일**: 이 노트북과 **같은 폴더**에 `2023_Simulator…` 로 시작하는 `.xlsx` 파일  
> **실행 방법**: 아래 `STEP 1` ~ `STEP 4` 셀만 수정 → **Run All Cells**

---

[Gitlab 데이터사이언스팀/kr_eeio](https://bidas-gitlab.boknet.intra/2620316/kr_eeio/-/tree/main/) 페이지에서 KR-EEIO 탄소배출 분석 파이프라인의 모든 코드와 데이터를 확인할 수 있음.

## ⚙️ 초기화 *(수정 불필요)*

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from eeio_simulator import load_eeio_data, run_simulation, save_excel, display_results
DATA = load_eeio_data()

📂 원본 파일: 2023_Simulator—일부원가만 아는 경우.xlsx
✅ 데이터 로딩 완료  (산업 33개)


---
## 📝 STEP 1. 기업 이름

In [2]:
COMPANY_NAME = "한은업체"    # ← 기업 이름을 입력하세요 (결과 파일명에 사용)

---
## 📝 STEP 2. 소속 산업 코드

| 코드 | 산업명 | 코드 | 산업명 |
|------|--------|------|--------|
| `A` | 농림수산품 | `J` | 정보통신·방송서비스 |
| `B` | 광산품 | `K` | 금융·보험서비스 |
| `C01` | 음식료품 | `L` | 부동산서비스 |
| `C02` | 섬유 및 가죽제품 | `M` | 전문·과학·기술서비스 |
| `C03` | 목재 및 종이, 인쇄 | `N` | 사업지원서비스 |
| `C04` | 석탄 및 석유제품 | `O` | 공공행정·국방·사회보장 |
| `C05` | 화학제품 | `P` | 교육서비스 |
| `C06` | 비금속광물제품 | `Q` | 보건·사회복지서비스 |
| `C07` | 1차 금속제품 | `R` | 예술·스포츠·여가서비스 |
| `C08` | 금속가공제품 | `S` | 기타서비스 |
| `C09` | 컴퓨터·전자·광학기기 | `T` | 기타 |
| `C10` | 전기장비 | `D` | 전력·가스·증기 |
| `C11` | 기계 및 장비 | `E` | 수도·폐기물처리·재활용 |
| `C12` | 운송장비 | `F` | 건설 |
| `C13` | 기타 제조업 제품 | `G` | 도소매·상품중개서비스 |
| `C14` | 제조임가공·산업용장비수리 | `H` | 운송서비스 |
| `I` | 음식점·숙박서비스 | | |

In [3]:
COMPANY_CODE = "C11"    # ← 위 표에서 해당 코드를 입력하세요

---
## 📝 STEP 3. 매출액

> 단위: **백만원** | 예) 10억 → `1_000` / 1조 → `1_000_000`

In [4]:
SALES = 1000    # ← 연간 매출액을 백만원 단위로 입력하세요

---
## 📝 STEP 4a. 원가 비중 (COST_RATIOS)

각 산업에서 구매하는 비용이 **매출 대비 몇 %** 인지 입력하세요.  
모르는 항목은 `0` 으로 두면 **국가평균으로 자동 추정**합니다.

> 💡 원가를 전혀 모르면 전부 `0` → Case 1 (매출액만 아는 경우)  
> 💡 일부만 알면 아는 항목만 입력 → Case 2 (일부원가만 아는 경우)  
> 💡 원가를 전부 알면 COST_RATIOS + VA_RATIOS 합계 = 100% → Case 3

In [5]:
COST_RATIOS = {
    "A":    0,    # 농림수산품
    "B":    0,    # 광산품
    "C01":  0,    # 음식료품
    "C02":  0,    # 섬유 및 가죽제품
    "C03":  5,    # 목재 및 종이, 인쇄
    "C04":  10,    # 석탄 및 석유제품
    "C05":  0,    # 화학제품
    "C06":  0,    # 비금속광물제품
    "C07":  0,    # 1차 금속제품
    "C08":  0,    # 금속가공제품
    "C09":  0,    # 컴퓨터·전자·광학기기
    "C10":  0,    # 전기장비
    "C11":  0,   # 기계 및 장비
    "C12":  0,    # 운송장비
    "C13":  0,    # 기타 제조업 제품
    "C14":  10,    # 제조임가공·산업용장비수리
    "D":    0,   # 전력·가스·증기
    "E":    0,    # 수도·폐기물처리·재활용
    "F":    0,    # 건설
    "G":    0,    # 도소매·상품중개서비스
    "H":    0,    # 운송서비스
    "I":    0,    # 음식점·숙박서비스
    "J":    0,    # 정보통신·방송서비스
    "K":    0,    # 금융·보험서비스
    "L":    0,    # 부동산서비스
    "M":    0,    # 전문·과학·기술서비스
    "N":    0,    # 사업지원서비스
    "O":    0,    # 공공행정·국방·사회보장
    "P":    0,    # 교육서비스
    "Q":    0,    # 보건·사회복지서비스
    "R":    0,    # 예술·스포츠·여가서비스
    "S":    0,    # 기타서비스
    "T":    0,    # 기타
}

---
## 📝 STEP 4b. 부가가치 비중 (VA_RATIOS)

인건비·영업이익·감가상각비가 **매출 대비 몇 %** 인지 입력하세요.  
모르면 `0` 으로 두면 국가평균으로 자동 추정합니다.

In [6]:
VA_RATIOS = {
    "피용자보수":   20,   # 인건비
    "영업잉여":     0,   # 영업이익
    "고정자본소모":  0,   # 감가상각비
}

---
## 🚀 계산 실행 *(수정 불필요)*

In [7]:
RESULT = run_simulation(DATA, {
    "company_name": COMPANY_NAME,
    "company_code": COMPANY_CODE,
    "sales":        SALES,
    "cost_ratios":  COST_RATIOS,
    "va_ratios":    VA_RATIOS,
})

---
## 📊 결과 확인 *(수정 불필요)*

In [8]:
display_results(RESULT)

항목,내용
기업명,한은업체
소속 산업,C11 기계 및 장비
매출액 (백만원),"1,000"
계산 모드,Case2 (일부원가만 아는 경우)


항목,배출량 (tCO2eq.),비중
Scope1 (직접배출),37.3,15.1%
Scope2 (전력 등 간접배출),67.5,27.3%
Scope3 (기타 간접배출),142.8,57.7%
합계 (Scope 1+2+3),247.6,100.0%


순위,코드,산업명,배출량 (tCO2eq.),비중
1,D,"전력, 가스 및 증기",67.5,27.3%
2,C07,1차 금속제품,57.2,23.1%
3,–,분석업체,37.3,15.1%
4,H,운송서비스,23.2,9.4%
5,C04,석탄 및 석유제품,22.1,8.9%
6,C05,화학제품,15.5,6.3%
7,E,"수도, 폐기물처리 및 재활용서비스",5.0,2.0%
8,C11,기계 및 장비,4.4,1.8%
9,C06,비금속광물제품,4.2,1.7%
10,C03,"목재 및 종이, 인쇄",3.4,1.4%


---
## 💾 결과 저장 *(수정 불필요)*

In [9]:
from pathlib import Path

DATA["output_dir"] = Path("/tmp/eeio_output")
DATA["output_dir"].mkdir(parents=True, exist_ok=True)

In [10]:
save_excel(RESULT, DATA["output_dir"])

💾 결과 저장: /tmp/eeio_output/한은업체_C11_탄소배출_20260728_153410.xlsx


PosixPath('/tmp/eeio_output/한은업체_C11_탄소배출_20260728_153410.xlsx')